
# Phase I: Book Cover Genre Classification with CLIP

**Model:** openai/clip-vit-base-patch32  
**Dataset:** BookCover30  
**Task:** Investigating genre bias and visual stereotypes in book cover classification.

In this phase, we analyze CLIP as a black-box model to understand how visual features and text prompts influence its genre predictions.



### 1.1 Dataset Structure

Check that the dataset directory exists and list the available files.


In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("archive")

print("Dataset folder exists:", DATA_DIR.exists())

for item in DATA_DIR.iterdir():
    print(item.name)



### 1.2 Dataset Metadata

Load the training and test metadata to inspect their structure, columns, and number of samples.


In [ ]:
train_df = pd.read_csv(DATA_DIR / "book30-listing-train.csv", sep=";")
test_df = pd.read_csv(DATA_DIR / "book30-listing-test.csv", sep=";")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())
display(test_df.head())

print("Columns:", train_df.columns.tolist())



### 1.3 Dataset Exploration

Explore the available book categories, their distribution, and the number of locally available cover images.


In [ ]:
# Explore categories
category_counts = train_df["Category"].value_counts()

print("Number of categories:", train_df["Category"].nunique())
display(category_counts.to_frame("Number of books"))

# Check local images
IMAGE_DIR = DATA_DIR / "title30cat"
image_files = list(IMAGE_DIR.rglob("*.jpg"))

print("\nTotal local images:", len(image_files))
print("Example image paths:")
for path in image_files[:5]:
    print(path)



### 1.4 Genre Selection

We selected six book categories with potentially distinctive visual characteristics to investigate whether CLIP relies on colors, illustrations, and common cover design patterns when predicting genres.


In [ ]:
selected_genres = [
    "Romance",
    "Mystery, Thriller & Suspense",
    "Science Fiction & Fantasy",
    "History",
    "Biographies & Memoirs",
    "Children's Books"
]

# Select 500 images per genre from the training split
selected_df = (
    train_df[train_df["Category"].isin(selected_genres)]
    .groupby("Category", group_keys=False)
    .sample(n=500, random_state=42)
    .reset_index(drop=True)
)

# Add local image paths
selected_df["image_path"] = selected_df["Filename"].apply(
    lambda name: str(IMAGE_DIR / "224x224" / name)
)

# Check image availability
selected_df["image_exists"] = selected_df["image_path"].apply(
    lambda path: Path(path).is_file()
)

print("Selected images:", len(selected_df))
print("Available images:", selected_df["image_exists"].sum())

display(selected_df["Category"].value_counts())



### 1.5 Visualizing Book Covers

Display sample book covers from each selected genre to explore potential visual patterns and formulate initial hypotheses about genre stereotypes.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(6, 5, figsize=(14, 19))

for row, genre in enumerate(selected_genres):
    samples = selected_df[
        selected_df["Category"] == genre
    ].sample(n=5, random_state=42)

    for col, (_, book) in enumerate(samples.iterrows()):
        img = Image.open(book["image_path"]).convert("RGB")

        axes[row, col].imshow(img)
        axes[row, col].axis("off")

        if col == 0:
            axes[row, col].set_ylabel(
                genre,
                fontsize=11,
                rotation=0,
                labelpad=95,
                va="center"
            )

plt.suptitle("Sample Book Covers by Genre", fontsize=16)
plt.tight_layout()
plt.show()



## 2. CLIP Model Setup

Load the pre-trained CLIP model for zero-shot book genre classification. We will compare each cover image with text descriptions of the six selected genres.


In [ ]:
%pip install torch transformers pillow tqdm

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

MODEL_NAME = "openai/clip-vit-base-patch32"

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

model.eval()

print("CLIP loaded successfully!")


### 2.1 First CLIP Predictions

Test CLIP on one book cover from each selected genre and compare its zero-shot predictions with the true labels.


In [ ]:
# Define text prompts for each genre
text_prompts = [
    f"A book cover of a {genre} book."
    for genre in selected_genres
]

# Select one image per genre
sample_books = (
    selected_df.groupby("Category", group_keys=False)
    .sample(n=1, random_state=42)
)

fig, axes = plt.subplots(1, 6, figsize=(18, 5))

for ax, (_, book) in zip(axes, sample_books.iterrows()):
    image = Image.open(book["image_path"]).convert("RGB")

    inputs = processor(
        text=text_prompts,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = outputs.logits_per_image.softmax(dim=1)

    predicted_idx = probabilities.argmax(dim=1).item()
    predicted_genre = selected_genres[predicted_idx]

    ax.imshow(image)
    ax.axis("off")
    ax.set_title(
        f"True: {book['Category']}\n"
        f"Pred: {predicted_genre}",
        fontsize=9,
        color="green" if predicted_genre == book["Category"] else "red"
    )

plt.tight_layout()
plt.show()


### 2.2 Genre Predictions on the Full Dataset

Run CLIP on all 3,000 book covers in batches and save the predicted genres and similarity scores for further analysis.


In [ ]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 16
all_probabilities = []

for start in tqdm(range(0, len(selected_df), batch_size)):
    batch = selected_df.iloc[start:start + batch_size]

    images = [
        Image.open(path).convert("RGB")
        for path in batch["image_path"]
    ]

    inputs = processor(
        text=text_prompts,
        images=images,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.inference_mode():
        outputs = model(**inputs)
        probs = outputs.logits_per_image.softmax(dim=1)

    all_probabilities.append(probs.cpu().numpy())

# Combine results
all_probabilities = np.concatenate(all_probabilities, axis=0)

# Save predictions
selected_df["predicted_genre"] = [
    selected_genres[i]
    for i in all_probabilities.argmax(axis=1)
]

selected_df["confidence"] = all_probabilities.max(axis=1)

for i, genre in enumerate(selected_genres):
    selected_df[f"prob_{i}"] = all_probabilities[:, i]

selected_df["correct"] = (
    selected_df["Category"] == selected_df["predicted_genre"]
)

# Save results
selected_df.to_csv("clip_predictions.csv", index=False)

print("Total predictions:", len(selected_df))
print("Accuracy:", round(selected_df["correct"].mean(), 3))
print("Results saved to clip_predictions.csv")


## 3. Model Behavior Analysis

### 3.1 Confusion Matrix

Visualize the relationship between true and predicted genres to identify systematic classification errors and frequently confused genre pairs.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(
    selected_df["Category"],
    selected_df["predicted_genre"],
    labels=selected_genres
)

fig, ax = plt.subplots(figsize=(12, 9))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=selected_genres
)

disp.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    xticks_rotation=45,
    colorbar=True
)

plt.title("CLIP Genre Classification — Confusion Matrix")
plt.xlabel("Predicted Genre")
plt.ylabel("True Genre")
plt.tight_layout()
plt.show()



### 3.2 Investigating History–Biography Confusion

Examine book covers that CLIP incorrectly classifies as Biographies & Memoirs instead of History, and compare them with correctly classified History covers.


In [ ]:

# History books incorrectly predicted as Biographies
history_errors = selected_df[
    (selected_df["Category"] == "History") &
    (selected_df["predicted_genre"] == "Biographies & Memoirs")
]

# Correctly predicted History books
history_correct = selected_df[
    (selected_df["Category"] == "History") &
    (selected_df["predicted_genre"] == "History")
]

print("History → Biography errors:", len(history_errors))
print("Correct History predictions:", len(history_correct))

fig, axes = plt.subplots(2, 6, figsize=(16, 8))

for row, (data, label) in enumerate([
    (history_errors, "Misclassified"),
    (history_correct, "Correct")
]):
    samples = data.sample(n=6, random_state=42)

    for col, (_, book) in enumerate(samples.iterrows()):
        image = Image.open(book["image_path"]).convert("RGB")
        axes[row, col].imshow(image)
        axes[row, col].axis("off")

        if col == 0:
            axes[row, col].set_ylabel(
                label,
                fontsize=12,
                rotation=90,
                labelpad=10
            )

plt.suptitle(
    "History Books: Misclassified vs Correctly Classified",
    fontsize=15
)
plt.tight_layout()
plt.show()



### 3.3 Testing the Face-Presence Hypothesis

Investigate whether History book covers containing detectable human faces are more frequently misclassified as Biographies & Memoirs.


In [ ]:
%pip install opencv-python

In [ ]:
import cv2
from tqdm.auto import tqdm

# Load a pre-trained face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

def has_face(image_path):
    image = cv2.imread(image_path)
    if image is None:
        return False

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=4,
        minSize=(20, 20)
    )

    return len(faces) > 0

# Analyze all History covers
history_df = selected_df[
    selected_df["Category"] == "History"
].copy()

history_df["has_face"] = [
    has_face(path)
    for path in tqdm(history_df["image_path"])
]

print("Total History covers:", len(history_df))
print("Covers with detected faces:", history_df["has_face"].sum())

display(pd.crosstab(
    history_df["has_face"],
    history_df["predicted_genre"] == "Biographies & Memoirs",
    rownames=["Face detected"],
    colnames=["Predicted as Biography"]
))


### 3.4 Face Presence and Genre Confusion

Compare the proportion of History covers predicted as Biographies & Memoirs between images with and without detected faces.


In [ ]:
# Calculate biography prediction rates
history_df["predicted_bio"] = (
    history_df["predicted_genre"] == "Biographies & Memoirs"
)

face_rates = (
    history_df.groupby("has_face")["predicted_bio"]
    .mean() * 100
)

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    ["No face detected", "Face detected"],
    [face_rates[False], face_rates[True]],
    color=["steelblue", "coral"]
)

for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{bar.get_height():.1f}%",
        ha="center",
        fontsize=12
    )

ax.set_ylabel("Predicted as Biography (%)")
ax.set_title("Does Face Presence Influence CLIP Predictions?")
ax.set_ylim(0, 65)

plt.tight_layout()
plt.show()



**Finding 1 — Face Presence and Genre Confusion**

Among History book covers, CLIP predicted Biographies & Memoirs for 54.5% of images with detected faces, compared to 23.3% without detected faces.

This suggests an association between visible faces and genre confusion. However, the face detector may miss small, illustrated, or historical faces, so the result does not establish causation.



### 3.5 Cover Brightness and Genre Confusion

Investigate whether darker Romance book covers are more likely to be classified as Mystery, Thriller & Suspense.

We measure the average brightness of each cover and compare prediction rates across brightness groups.


In [ ]:

import numpy as np
from PIL import Image

# Calculate average grayscale brightness (0–255)
def get_brightness(image_path):
    with Image.open(image_path) as img:
        gray = img.convert("L")
        return np.asarray(gray).mean()

# Select Romance covers
romance_df = selected_df[
    selected_df["Category"] == "Romance"
].copy()

romance_df["brightness"] = romance_df["image_path"].apply(
    get_brightness
)

romance_df["predicted_thriller"] = (
    romance_df["predicted_genre"] ==
    "Mystery, Thriller & Suspense"
)

# Divide covers into four equally sized brightness groups
romance_df["brightness_group"] = pd.qcut(
    romance_df["brightness"],
    q=4,
    labels=["Darkest", "Dark", "Bright", "Brightest"]
)

# Calculate Thriller prediction rate for each group
brightness_results = (
    romance_df.groupby("brightness_group", observed=True)
    ["predicted_thriller"]
    .agg(["count", "sum", "mean"])
)

brightness_results["mean"] *= 100

display(
    brightness_results.rename(columns={
        "count": "Total covers",
        "sum": "Predicted as Thriller",
        "mean": "Thriller prediction (%)"
    }).round(1)
)



### 3.6 Brightness and Genre Misclassification

Visualize how the proportion of Romance covers misclassified as Mystery, Thriller & Suspense changes across four brightness groups.


In [ ]:
labels = ["Darkest", "Dark", "Bright", "Brightest"]
rates = brightness_results.loc[labels, "mean"]

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    labels,
    rates,
    color=["#34324A", "#67627F", "#A9A5C4", "#DCD9ED"]
)

for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{bar.get_height():.1f}%",
        ha="center",
        fontsize=11
    )

ax.set_ylabel("Predicted as Thriller (%)")
ax.set_xlabel("Cover Brightness Group")
ax.set_title("Romance Covers: Brightness vs Thriller Predictions")
ax.set_ylim(0, 28)

plt.tight_layout()
plt.show()



**Finding 2 — Cover Brightness and Genre Confusion**

Among 500 Romance covers, the proportion classified as Mystery, Thriller & Suspense decreased consistently with increasing brightness:

- Darkest: 23.2%
- Dark: 16.8%
- Bright: 10.4%
- Brightest: 4.0%

The darkest covers were 5.8 times more likely to be classified as Thriller than the brightest covers.

This supports an association between cover brightness and genre predictions, but does not establish that brightness itself causes the classification errors.



## 4. Prompt Sensitivity Analysis

### 4.1 Comparing Prompt Templates

Investigate whether different text descriptions of book genres influence CLIP's predictions.

We compare a neutral genre description with a more descriptive prompt containing common genre-related themes and visual stereotypes.


In [ ]:
# Original neutral prompts
neutral_prompts = [
    f"A book cover of a {genre} book."
    for genre in selected_genres
]

# More descriptive genre prompts
descriptive_prompts = [
    "A book cover of a romantic love story novel.",
    "A book cover of a mysterious crime thriller novel.",
    "A book cover of a science fiction or fantasy adventure novel.",
    "A book cover of a historical nonfiction book.",
    "A book cover of a biography about a person's life.",
    "A book cover of a children's illustrated storybook."
]

# Display both prompt sets
for genre, neutral, descriptive in zip(
    selected_genres, neutral_prompts, descriptive_prompts
):
    print(f"\nGenre: {genre}")
    print(f"Neutral:     {neutral}")
    print(f"Descriptive: {descriptive}")



### 4.2 Predictions with Descriptive Prompts

Run CLIP on the same 3,000 book covers using descriptive genre prompts. Compare the new predictions with the original neutral-prompt baseline.


In [ ]:
# Run CLIP with descriptive prompts
descriptive_probabilities = []

for start in tqdm(range(0, len(selected_df), batch_size)):
    batch = selected_df.iloc[start:start + batch_size]

    images = []
    for path in batch["image_path"]:
        with Image.open(path) as img:
            images.append(img.convert("RGB"))

    inputs = processor(
        text=descriptive_prompts,
        images=images,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.inference_mode():
        outputs = model(**inputs)
        probs = outputs.logits_per_image.softmax(dim=1)

    descriptive_probabilities.append(probs.cpu().numpy())

descriptive_probabilities = np.concatenate(
    descriptive_probabilities, axis=0
)

# Store new predictions
selected_df["descriptive_prediction"] = [
    selected_genres[i]
    for i in descriptive_probabilities.argmax(axis=1)
]

selected_df["descriptive_correct"] = (
    selected_df["Category"] == selected_df["descriptive_prediction"]
)

# Compare results
neutral_accuracy = selected_df["correct"].mean()
descriptive_accuracy = selected_df["descriptive_correct"].mean()

changed_predictions = (
    selected_df["predicted_genre"] !=
    selected_df["descriptive_prediction"]
).sum()

print(f"Neutral accuracy: {neutral_accuracy:.2%}")
print(f"Descriptive accuracy: {descriptive_accuracy:.2%}")
print(f"Changed predictions: {changed_predictions} / {len(selected_df)}")
print(f"Prediction change rate: {changed_predictions / len(selected_df):.2%}")

# Save updated results
selected_df.to_csv("clip_predictions.csv", index=False)



### 4.3 Classification Accuracy by Genre

Compare classification accuracy for each genre using neutral and descriptive prompts.

We investigate which genres benefit from descriptive prompts and which become more difficult for CLIP to classify.


In [ ]:
# Compare accuracy for each genre
genre_comparison = (
    selected_df.groupby("Category")
    .agg(
        neutral_accuracy=("correct", "mean"),
        descriptive_accuracy=("descriptive_correct", "mean")
    )
    * 100
)

genre_comparison["difference"] = (
    genre_comparison["descriptive_accuracy"]
    - genre_comparison["neutral_accuracy"]
)

display(genre_comparison.round(2))

# Visualize the comparison
ax = genre_comparison[
    ["neutral_accuracy", "descriptive_accuracy"]
].plot(
    kind="bar",
    figsize=(11, 6),
    width=0.75
)

ax.set_title("CLIP Accuracy by Genre: Neutral vs Descriptive Prompts")
ax.set_xlabel("Book Genre")
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(0, 100)
ax.legend(["Neutral", "Descriptive"])

plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()



### 4.4 Changes in Genre Predictions

Analyze how CLIP's predictions shift between neutral and descriptive prompts.

We focus on which genres gain or lose predictions and identify the most common transitions between predicted genres.


In [ ]:
# Count predictions for each genre
prediction_counts = pd.DataFrame({
    "Neutral": selected_df["predicted_genre"].value_counts(),
    "Descriptive": selected_df["descriptive_prediction"].value_counts()
}).reindex(selected_genres).fillna(0).astype(int)

prediction_counts["Difference"] = (
    prediction_counts["Descriptive"] - prediction_counts["Neutral"]
)

display(prediction_counts)

# Visualize how prediction counts change
ax = prediction_counts[["Neutral", "Descriptive"]].plot(
    kind="bar",
    figsize=(11, 5),
    width=0.75
)

ax.set_title("How Prompt Wording Changes CLIP's Genre Predictions")
ax.set_xlabel("Predicted Genre")
ax.set_ylabel("Number of Covers")
ax.legend(title="Prompt Type")

plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

# Most frequent changes between predicted genres
changed_df = selected_df[
    selected_df["predicted_genre"] !=
    selected_df["descriptive_prediction"]
]

transitions = (
    changed_df.groupby(
        ["predicted_genre", "descriptive_prediction"]
    )
    .size()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name="Count")
)

display(transitions)



### 4.5 Corrected Predictions vs New Errors

Investigate whether changing the prompts corrects previous classification errors or introduces new ones.

We also examine how often books from other genres are incorrectly classified as History after switching to descriptive prompts.


In [ ]:
# Compare correctness before and after changing prompts
selected_df["prediction_outcome"] = np.select(
    [
        selected_df["correct"] & selected_df["descriptive_correct"],
        ~selected_df["correct"] & selected_df["descriptive_correct"],
        selected_df["correct"] & ~selected_df["descriptive_correct"],
        ~selected_df["correct"] & ~selected_df["descriptive_correct"]
    ],
    [
        "Correct in both",
        "Corrected by descriptive",
        "New error from descriptive",
        "Wrong in both"
    ],
    default="Unknown"
)

outcome_counts = selected_df["prediction_outcome"].value_counts()

display(outcome_counts.to_frame("Number of covers"))

# Analyze History predictions
history_analysis = selected_df[
    selected_df["descriptive_prediction"] == "History"
]

print("Total predicted as History:", len(history_analysis))
print(
    "Actually History:",
    (history_analysis["Category"] == "History").sum()
)
print(
    "Actually another genre:",
    (history_analysis["Category"] != "History").sum()
)

# Plot prediction outcomes
plt.figure(figsize=(10, 5))

order = [
    "Correct in both",
    "Corrected by descriptive",
    "New error from descriptive",
    "Wrong in both"
]

bars = plt.bar(
    order,
    outcome_counts.reindex(order, fill_value=0),
    color=["#4C9F70", "#78B7D0", "#E77C70", "#A0A0A0"]
)

plt.bar_label(bars, padding=3)
plt.title("Impact of Descriptive Prompts on CLIP Predictions")
plt.ylabel("Number of Book Covers")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()



### 4.6 Visual Inspection of Changed Predictions

Compare book covers where descriptive prompts corrected previous errors with covers where they introduced new errors.

We inspect whether the affected covers share visual characteristics such as colors, typography, portraits, or illustrations.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def show_prediction_examples(outcome, n=6):
    examples = selected_df[
        selected_df["prediction_outcome"] == outcome
    ].sample(
        n=min(n, (selected_df["prediction_outcome"] == outcome).sum()),
        random_state=42
    )

    fig, axes = plt.subplots(1, len(examples), figsize=(16, 4))

    for ax, (_, book) in zip(axes, examples.iterrows()):
        with Image.open(book["image_path"]) as img:
            ax.imshow(img.convert("RGB"))

        ax.set_title(
            f"True: {book['Category']}\n"
            f"Neutral: {book['predicted_genre']}\n"
            f"Descriptive: {book['descriptive_prediction']}",
            fontsize=8
        )
        ax.axis("off")

    plt.suptitle(outcome, fontsize=14)
    plt.tight_layout()
    plt.show()

show_prediction_examples("Corrected by descriptive")
show_prediction_examples("New error from descriptive")



### 4.7 Investigating History Prediction Bias

Descriptive prompts substantially increased the number of books classified as History.

We investigate which true genres contribute to these predictions and whether the increase mainly reflects corrected predictions or new classification errors.


In [ ]:
# Compare true genres among books predicted as History
history_comparison = pd.DataFrame({
    "Neutral": selected_df[
        selected_df["predicted_genre"] == "History"
    ]["Category"].value_counts(),

    "Descriptive": selected_df[
        selected_df["descriptive_prediction"] == "History"
    ]["Category"].value_counts()
}).reindex(selected_genres, fill_value=0).fillna(0).astype(int)

history_comparison["Difference"] = (
    history_comparison["Descriptive"] -
    history_comparison["Neutral"]
)

display(history_comparison)

# Plot the composition of History predictions
ax = history_comparison[
    ["Neutral", "Descriptive"]
].plot(
    kind="bar",
    figsize=(11, 5),
    width=0.75
)

ax.set_title("True Genres of Books Predicted as History")
ax.set_xlabel("Actual Book Genre")
ax.set_ylabel("Number of Covers Predicted as History")
ax.legend(title="Prompt Type")

plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


**Observation:** Descriptive prompts strongly increase the number of books predicted as History, from 486 to 1,168.

Although correct History predictions increase from 297 to 426, incorrect History predictions also increase substantially, from 189 to 742.

The largest increases in misclassification come from Biographies & Memoirs, Mystery/Thriller, and Romance.

This suggests that CLIP's predictions are highly sensitive to prompt wording. The descriptive prompts introduce a strong prediction bias toward History, even for books belonging to other genres.


In [ ]:
import pandas as pd

results_df = pd.read_csv("clip_predictions.csv")

print("Total books:", len(results_df))
print("\nAvailable columns:")
print(results_df.columns.tolist())

display(results_df.head())


## Interactive Dashboard

We created an interactive dashboard using **Streamlit**, a Python library for building web applications.

The dashboard allows users to explore CLIP's book genre classification results without running the experiments again. It uses the predictions saved in `clip_predictions.csv` and displays them in an interactive web interface.

### Dashboard Features

The dashboard contains five main sections:

- **Model Overview:** Shows the total number of book covers and the overall classification accuracy for neutral and descriptive prompts.
- **Genre Analysis:** Allows users to select a genre and compare the accuracy of both prompt types.
- **Error Explorer:** Displays misclassified book covers and allows users to filter them by predicted genre and prompt type.
- **Prompt Comparison Explorer:** Shows examples where changing the text prompt changed CLIP's prediction, including corrected predictions and new errors.
- **Confidence Analysis:** Compares confidence scores for correct and incorrect predictions and allows users to investigate high-confidence mistakes.

Users can interact with the dashboard using dropdown menus, sliders, and other controls. The visualizations and displayed examples update automatically based on the selected filters.

### Installation and Setup

To run the dashboard, Python and Streamlit are required.

First, install Streamlit by opening a terminal and running:

```bash
python3 -m pip install streamlit
```

Make sure the following files are available:

- `dashboard.py` — the Streamlit application
- `clip_predictions.csv` — the saved CLIP classification results
- `archive/` — the dataset folder containing the book cover images

The dataset images must be accessible through the paths stored in the CSV file.

### Running the Dashboard

Open a terminal and navigate to the folder containing `dashboard.py`.

For example, if the file is on the Desktop:

```bash
cd ~/Desktop
```

Then start the application:

```bash
python3 -m streamlit run dashboard.py
```

Streamlit will start a local web server. The dashboard can usually be opened in a browser at:

http://localhost:8501

The application runs locally and does not require retraining CLIP.

To stop the dashboard, return to the terminal and press `Ctrl + C`.

### Purpose

The dashboard helps us explore the results of Phase I in a more interactive way. Instead of looking only at static plots, we can filter predictions, inspect individual book covers, and compare different model behaviors.

This makes it easier to identify interesting patterns and investigate classification errors.


## 5. Confidence Analysis

In this section, we investigate how confident CLIP is in its genre predictions.

We compare confidence scores for correct and incorrect predictions and explore cases where the model makes mistakes with high confidence.

Our goal is to understand whether CLIP's confidence is related to its classification accuracy.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load saved predictions
df = pd.read_csv("clip_predictions.csv")

# Confidence is stored between 0 and 1
df["confidence"] = pd.to_numeric(df["confidence"], errors="coerce")

# Compare correct and incorrect predictions
confidence_summary = (
    df.groupby("correct")["confidence"]
    .agg(["mean", "median", "count"])
    .rename(index={True: "Correct", False: "Incorrect"})
)

display(confidence_summary)

# Visualize confidence distributions
plt.figure(figsize=(10, 5))

plt.hist(
    df.loc[df["correct"] == True, "confidence"].dropna(),
    bins=25,
    alpha=0.6,
    density=True,
    label="Correct"
)

plt.hist(
    df.loc[df["correct"] == False, "confidence"].dropna(),
    bins=25,
    alpha=0.6,
    density=True,
    label="Incorrect"
)

plt.xlabel("CLIP Confidence")
plt.ylabel("Density")
plt.title("Confidence Distribution: Correct vs Incorrect Predictions")
plt.legend()
plt.tight_layout()
plt.show()



### High-Confidence Errors

We investigate how often CLIP makes incorrect predictions at different confidence thresholds.

We use thresholds of 70%, 80%, 90%, and 95% to see whether higher confidence is associated with fewer classification errors.

We also examine whether the model can still make mistakes when it is highly confident.


In [ ]:
# Analyze errors at different confidence thresholds

thresholds = [0.7, 0.8, 0.9, 0.95]
results = []

for threshold in thresholds:
    subset = df[df["confidence"] >= threshold]

    total = len(subset)
    incorrect = (~subset["correct"].astype(bool)).sum()

    error_rate = (
        incorrect / total * 100 if total > 0 else 0
    )

    results.append({
        "Confidence threshold": f"{threshold:.0%}",
        "Total predictions": total,
        "Incorrect predictions": incorrect,
        "Error rate (%)": round(error_rate, 2)
    })

confidence_results = pd.DataFrame(results)

display(confidence_results)



### Confidence Analysis — Findings

We found that CLIP generally makes fewer mistakes when its confidence is higher.

For predictions with confidence of at least 70%, the error rate was 19.83%. This decreased to 10.84% at 90% confidence and 7.25% at 95% confidence.

However, high confidence does not always mean that a prediction is correct. Even at 95% confidence, CLIP incorrectly classified 15 book covers.

This suggests that confidence is useful for identifying more reliable predictions, but the model can still make confident mistakes.



## Final Research Findings — Phase I

### 1. Genre Classification Performance

We evaluated CLIP on 3,000 book covers from six genres, with 500 images per genre.

Using neutral text prompts, CLIP achieved an overall accuracy of 62.3%.

The model performed best on Romance (73.2%) and Children's Books (68.2%). Science Fiction & Fantasy was the most difficult genre, with an accuracy of 51.4%.

The confusion matrix showed that some genres are frequently confused. For example, 127 History books were classified as Biographies & Memoirs.

This suggests that book covers from different genres can share similar visual characteristics.

### 2. Visual Features and Classification Errors

We investigated whether visual features of book covers are associated with CLIP's predictions.

For History books, we tested whether the presence of a detected face was related to predictions of Biographies & Memoirs.

When a face was detected, 54.5% of covers were classified as Biographies. Without a detected face, this happened in 23.3% of cases.

We also investigated brightness in Mystery, Thriller & Suspense covers.

Among the darkest covers, 23.2% were predicted as Thriller, compared with only 4.0% among the brightest covers.

These results suggest that CLIP's predictions are associated with visual cues such as faces and brightness. However, these observations do not prove that these features directly cause the predictions.

### 3. Sensitivity to Prompt Wording

We compared two types of text prompts: neutral genre descriptions and more descriptive prompts.

Neutral prompts achieved 62.3% accuracy, while descriptive prompts achieved 52.3%.

Changing the prompts changed 1,199 out of 3,000 predictions (39.97%).

Descriptive prompts improved performance for History and Science Fiction & Fantasy, but reduced accuracy for several other genres.

One interesting finding was that the number of books predicted as History increased from 486 to 1,168.

This shows that CLIP is sensitive to how genre descriptions are written. More descriptive prompts do not necessarily produce better predictions.

### 4. Corrected Predictions and New Errors

We investigated how descriptive prompts affected individual predictions.

Descriptive prompts corrected 274 previously incorrect predictions, but also introduced 575 new errors.

Another 1,295 predictions were correct with both prompt types, while 856 remained incorrect.

We also examined example book covers to understand these changes.

The results show that changing prompt wording can improve some predictions while making others worse.

### 5. Confidence and Model Reliability

We analyzed CLIP's confidence scores for correct and incorrect predictions.

Correct predictions had a higher average confidence (0.717) than incorrect predictions (0.559).

The error rate decreased as we selected predictions with higher confidence.

For predictions with at least 90% confidence, the error rate was 10.84%. At 95% confidence, the error rate decreased to 7.25%.

However, 15 predictions with at least 95% confidence were still incorrect.

This shows that confidence is related to prediction correctness, but high confidence does not guarantee that CLIP is correct.

### Overall Conclusion

Our experiments show that CLIP can recognize visual patterns associated with book genres, but its predictions have several limitations.

The model sometimes confuses visually similar genres, its predictions are associated with visual characteristics of book covers, and its performance changes considerably depending on prompt wording.

We also found that CLIP can make incorrect predictions with high confidence.

Overall, our findings suggest that CLIP's behavior depends on both the visual input and the text descriptions used for classification.
